# Hiraeth — QLoRA Fine-tune on Kaggle (2x T4 / P100)

Pipeline: `DMJ-Dataset-Builder` output -> chat-formatted JSONL -> QLoRA SFT on `Qwen2.5-7B-Instruct` -> merge adapter -> test.

**Before running:** In Kaggle, set **Settings > Accelerator > GPU T4 x2** (or P100 x2), and enable **Internet** (needed to pull the base model + your dataset repo).

If your DMJ dataset repo/output is private or on your own machine, upload the final merged `.jsonl` as a Kaggle Dataset and adjust the path below instead of cloning.

In [ ]:
!nvidia-smi

## 1. Install dependencies

In [ ]:
!pip install -q -U transformers datasets peft trl bitsandbytes accelerate sentencepiece protobuf huggingface_hub

## 2. Get the DMJ Dataset Builder output

Clone your dataset-builder repo and run its pipeline, OR skip straight to step 3 if you already have `final_merged.jsonl` (e.g. uploaded as a Kaggle input dataset at `/kaggle/input/your-dataset/final_merged.jsonl`).

In [ ]:
!git clone https://github.com/jadhavdurvesh/DMJ-Dataset-Builder.git
%cd DMJ-Dataset-Builder
!pip install -q -r requirements.txt
!python build.py download
!python build.py convert
!python build.py validate
!python build.py merge
%cd ..

In [ ]:
# Point this at wherever build.py merge wrote the final dataset
# (check DMJ-Dataset-Builder/datasets/final/ for the exact filename)
!ls DMJ-Dataset-Builder/datasets/final/

## 3. Get the Hiraeth training scripts

Upload `prepare_dataset.py`, `train.py`, `merge_and_save.py`, `chat.py` as a Kaggle Dataset/Notebook input,
or paste them into cells here. This example assumes they're in `/kaggle/working/scripts/`.

In [ ]:
!mkdir -p /kaggle/working/scripts /kaggle/working/data

## 4. Prepare the dataset (DMJ schema -> chat format)

In [ ]:
!python /kaggle/working/scripts/prepare_dataset.py \
    --input DMJ-Dataset-Builder/datasets/final/<REPLACE_WITH_FINAL_FILENAME>.jsonl \
    --output_dir /kaggle/working/data \
    --val_split 0.02 \
    --system_prompt "You are Hiraeth, a helpful, precise AI assistant."

## 5. Train (QLoRA, sharded across both GPUs)

In [ ]:
!python /kaggle/working/scripts/train.py \
    --base_model Qwen/Qwen2.5-7B-Instruct \
    --train_file /kaggle/working/data/train.jsonl \
    --val_file /kaggle/working/data/val.jsonl \
    --output_dir /kaggle/working/hiraeth-qlora \
    --num_train_epochs 3 \
    --per_device_train_batch_size 2 \
    --gradient_accumulation_steps 8 \
    --learning_rate 2e-4 \
    --max_seq_length 2048

## 6. Merge adapter into a standalone model

In [ ]:
!python /kaggle/working/scripts/merge_and_save.py \
    --base_model Qwen/Qwen2.5-7B-Instruct \
    --adapter_dir /kaggle/working/hiraeth-qlora \
    --output_dir /kaggle/working/hiraeth-merged

## 7. Quick sanity chat

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_dir = "/kaggle/working/hiraeth-merged"
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForCausalLM.from_pretrained(model_dir, torch_dtype=torch.bfloat16, device_map="auto")

messages = [
    {"role": "system", "content": "You are Hiraeth, a helpful, precise AI assistant."},
    {"role": "user", "content": "Write a Python function that reverses a linked list."},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=400, do_sample=True, temperature=0.7, top_p=0.9)
print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))